## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

plt.rcParams['figure.dpi'] = 110

print('Imports OK')
print(f'LightGBM: {lgb.__version__}')
print(f'Optuna:   {optuna.__version__}')

## 1. Carga de datos

In [ ]:
#df = pd.read_csv('dataset_7000_53.csv')

df = pd.read_csv("dataset_4913_53.csv") #dataset con target sin NA

print(f'Shape: {df.shape}')
print(f"n_shows_24_25 — con dato: {df['n_shows_24_25'].notna().sum()}  "
      f"sin dato: {df['n_shows_24_25'].isna().sum()}")
df[['artist_name','sp_monthly_listeners','ins_followers','n_shows_24_25']].head(3)

Shape: (7000, 53)
n_shows_24_25 — con dato: 4913  sin dato: 2087


,artist_name,sp_monthly_listeners,ins_followers,n_shows_24_25
0,Bad Bunny,112734492.0,54721804.0,45.0
1,Taylor Swift,102733014.0,280531827.0,49.0
2,Bruno Mars,134166866.0,42922145.0,55.0


# Feature engineering: ratios

In [ ]:
# CREAR 

# 'ins_followers'/ 'sp_monthly_listeners'

# 'tiktok_sp_ratio' / 'sp_monthly_listeners'

# 'youtube_sp_ratio' / 'ycs_views'

# dimensiones variables
creo tabla

In [ ]:
# Agrupación conceptual de variables
# Construcción de diccionario de variables

# ¿agrupo los ratios del feature engineering?

id_cols = [
    "chartmetric_id",
    "artist_name"
]

metadata_cols = [
    "country",
    "country_short",
    "pronoun_title",
    "pronoun_short",
    "record_label",
    "major_record_label",
    "primary_genre",
    "genre_short",
    "band",
    "muerto_disuelto"
]

streaming_cols = [
    "sp_followers",
    "sp_monthly_listeners",
    "sp_popularity",
    "deezer_fans",
    "shazam_count",
    "pandora_lifetime_streams",
    "pandora_lifetime_stations_added"
]

playlist_cols = [
    "sp_playlist_total_reach",
    "num_sp_editorial_playlists",
    "num_sp_playlists",
    "sp_editorial_playlist_total_reach",
    "num_am_editorial_playlists",
    "num_am_playlists",
    "num_de_editorial_playlists",
    "num_de_playlists",
    "de_playlist_total_reach",
    "de_editorial_playlist_total_reach",
    "num_az_editorial_playlists",
    "num_az_playlists",
    "num_yt_editorial_playlists",
    "num_yt_playlists",
    "yt_playlist_total_reach"
]

social_cols = [
    "ins_followers",
    "twitter_followers",
    "tiktok_followers",
    "tiktok_likes",
    "tiktok_top_video_views",
    "tiktok_top_video_comments",
    "tiktok_track_posts"
]

youtube_cols = [
    "ycs_subscribers",
    "ycs_views",
    "youtube_daily_video_views",
    "youtube_monthly_video_views"
]

live_cols = [
    "n_shows_24_25",
    "n_shows_with_capacity_24_25",
    "capacity_24_25",
    "avg_venue_capacity_24_25",
    "n_cities_24_25",
    "n_countries_24_25",
    "shows_per_country_24_25",
    "off_cycle"
]


# Diccionario de dimensiones
column_groups = {
    "identificacion": id_cols,
    "metadata": metadata_cols,
    "streaming": streaming_cols,
    "playlists": playlist_cols,
    "redes_sociales": social_cols,
    "youtube": youtube_cols,
    "live_events": live_cols
}

# ============================================================
# Tabla auxiliar con identidad conceptual de cada variable
# ============================================================

variable_dimension = []

for dimension, cols in column_groups.items():
    for col in cols:
        variable_dimension.append({
            "variable": col,
            "dimension": dimension
        })

df_variable_dimension = pd.DataFrame(variable_dimension)

# ============================================================
# Control de cobertura de columnas
# ============================================================

grouped_cols = df_variable_dimension["variable"].tolist()

cols_in_groups_not_in_df = sorted(set(grouped_cols) - set(df.columns))
cols_in_df_not_in_groups = sorted(set(df.columns) - set(grouped_cols))

print("Columnas del df:", df.shape[1])
print("Columnas agrupadas:", len(grouped_cols))
print("Columnas agrupadas que no están en df:", cols_in_groups_not_in_df)
print("Columnas del df no agrupadas:", cols_in_df_not_in_groups)

# ============================================================
# Resumen por dimensión
# ============================================================

resumen_dimensiones = (
    df_variable_dimension
    .groupby("dimension", as_index=False)
    .agg(n_variables=("variable", "count"))
    .sort_values("n_variables", ascending=False)
)

display(resumen_dimensiones)

# ============================================================
# Vista del diccionario de variables
# ============================================================

display(df_variable_dimension)

Columnas del df: 53
Columnas agrupadas: 53
Columnas agrupadas que no están en df: []
Columnas del df no agrupadas: []


,dimension,n_variables
3,playlists,15
2,metadata,10
1,live_events,8
5,streaming,7
4,redes_sociales,7
6,youtube,4
0,identificacion,2


,variable,dimension
0,chartmetric_id,identificacion
1,artist_name,identificacion
2,country,metadata
3,country_short,metadata
4,pronoun_title,metadata
5,pronoun_short,metadata
6,record_label,metadata
7,major_record_label,metadata
8,primary_genre,metadata
9,genre_short,metadata


# tipos datos

In [ ]:
# 1. Definir el diccionario de tipos

# agregar los ratios en el diccionario

dtypes_dict = {
    'chartmetric_id': 'Int64',
    'artist_name': 'str',
    'country': 'category',
    'pronoun_title': 'category',
    'record_label': 'category',
    'primary_genre': 'category',
    'sp_followers': 'Int64',
    'sp_monthly_listeners': 'Int64',
    'sp_popularity': 'float64',
    'sp_playlist_total_reach': 'float64',
    'ins_followers': 'Int64',
    'twitter_followers': 'Int64',
    'tiktok_followers': 'Int64',
    'tiktok_likes': 'Int64',
    'ycs_subscribers': 'Int64',
    'ycs_views': 'Int64',
    'youtube_daily_video_views': 'Int64',
    'youtube_monthly_video_views': 'Int64',
    'deezer_fans': 'Int64',
    'shazam_count': 'Int64',
    'pandora_lifetime_streams': 'Int64',
    'pandora_lifetime_stations_added': 'Int64',
    'band': 'bool',
    'num_sp_editorial_playlists': 'Int64',
    'num_sp_playlists': 'Int64',
    'sp_editorial_playlist_total_reach': 'float64',
    'num_am_editorial_playlists': 'Int64',
    'num_am_playlists': 'Int64',
    'num_de_editorial_playlists': 'Int64',
    'num_de_playlists': 'Int64',
    'de_playlist_total_reach': 'float64',
    'de_editorial_playlist_total_reach': 'float64',
    'num_az_editorial_playlists': 'Int64',
    'num_az_playlists': 'Int64',
    'num_yt_editorial_playlists': 'Int64',
    'num_yt_playlists': 'Int64',
    'yt_playlist_total_reach': 'float64',
    'tiktok_top_video_views': 'float64',
    'tiktok_top_video_comments': 'float64',
    'tiktok_track_posts': 'float64',
    'n_shows_24_25': 'Int64',
    'n_shows_with_capacity_24_25': 'Int64',
    'capacity_24_25': 'float64',
    'avg_venue_capacity_24_25': 'float64',
    'n_cities_24_25': 'Int64',
    'n_countries_24_25': 'Int64',
    'shows_per_country_24_25': 'float64',
    'major_record_label': 'boolean',
    'pronoun_short': 'category',
    'genre_short': 'category',
    'country_short': 'category',
    'off_cycle': 'bool',
    'muerto_disuelto': 'boolean'
}

# Aplicar la conversión al dataframe df

# Conversión solo de columnas existentes
dtypes_presentes = {
    col: dtype
    for col, dtype in dtypes_dict.items()
    if col in df.columns
}

df = df.astype(dtypes_presentes)

# 3. Verificación rápida
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4913 entries, 0 to 4912
Data columns (total 53 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   chartmetric_id                     4913 non-null   Int64   
 1   artist_name                        4913 non-null   str     
 2   country                            4909 non-null   category
 3   pronoun_title                      4913 non-null   category
 4   record_label                       4604 non-null   category
 5   primary_genre                      4913 non-null   category
 6   sp_followers                       4907 non-null   Int64   
 7   sp_monthly_listeners               4907 non-null   Int64   
 8   sp_popularity                      4911 non-null   float64 
 9   sp_playlist_total_reach            4913 non-null   float64 
 10  ins_followers                      4580 non-null   Int64   
 11  twitter_followers                  3968 non-null   Int

In [ ]:
# Estructura general del dataset
print("Filas y columnas:", df.shape)

# Control de clave única
print("Artistas únicos:", df["chartmetric_id"].nunique())
print("Duplicados por chartmetric_id:", df.duplicated(subset="chartmetric_id").sum())

# Control de target
print("Nulos en target:", df["n_shows_24_25"].isna().sum())
print("Tipo de dato de target:", df["n_shows_24_25"].dtype)

Filas y columnas: (4913, 53)
Artistas únicos: 4913
Duplicados por chartmetric_id: 0
Nulos en target: 0
Tipo de dato de target: Int64


pasar a log ?


## 2. Feature Engineering

In [ ]:
# ── 2a. Target: log1p(n_shows_24_25) ─────────────────────────────────────────
TARGET = 'n_shows_24_25'
y_all = np.log1p(df[TARGET].astype(float).values)

print(f"Target shape : {y_all.shape}")
print(f"Target range : [{y_all.min():.3f}, {y_all.max():.3f}]")
print(f"Target mean  : {y_all.mean():.3f}  |  median: {np.median(y_all):.3f}")

In [ ]:
# ── 2b. Log1p transforms + ratio features ────────────────────────────────────
#
# LightGBM is tree-based (invariant to monotone transforms on features),
# but log1p compresses extreme tails and produces better histogram bins
# when max_bin=31 (coarse discretisation — bin quality matters here).
# sp_popularity is already 0-100: no transform needed.
# Boolean / category columns: no transform needed.

vars_log1p = (
    [c for c in streaming_cols  if c != 'sp_popularity']
    + playlist_cols
    + social_cols
    + youtube_cols
)

df_fe = df.copy()

for col in vars_log1p:
    df_fe[col] = np.log1p(df_fe[col].astype(float))

# Log-ratio features (log(a/b) = log(a) - log(b) — avoids division by zero)
df_fe['log_ins_sp_ratio']     = (np.log1p(df['ins_followers'].astype(float))
                                  - np.log1p(df['sp_monthly_listeners'].astype(float)))
df_fe['log_tiktok_sp_ratio']  = (np.log1p(df['tiktok_followers'].astype(float))
                                  - np.log1p(df['sp_monthly_listeners'].astype(float)))
df_fe['log_youtube_sp_ratio'] = (np.log1p(df['ycs_views'].astype(float))
                                  - np.log1p(df['sp_monthly_listeners'].astype(float)))

ratio_cols = ['log_ins_sp_ratio', 'log_tiktok_sp_ratio', 'log_youtube_sp_ratio']

print("Log1p applied to:", len(vars_log1p), "variables")
print("Ratio features created:", ratio_cols)

In [ ]:
# ── 2c. Feature matrix: campos_buenos ────────────────────────────────────────
#
# Exclude:
#   • identifiers            : chartmetric_id, artist_name
#   • target                 : n_shows_24_25
#   • leakage (live events)  : all other live_cols
#   • high-cardinality cols  : country, pronoun_title, record_label, primary_genre
#                              (replaced by _short reduced-cardinality versions)

LEAKAGE_COLS = [
    'n_shows_with_capacity_24_25', 'capacity_24_25', 'avg_venue_capacity_24_25',
    'n_cities_24_25', 'n_countries_24_25', 'shows_per_country_24_25', 'off_cycle',
]
HIGH_CARD_COLS = ['country', 'pronoun_title', 'record_label', 'primary_genre']
EXCLUDE_COLS   = id_cols + [TARGET] + LEAKAGE_COLS + HIGH_CARD_COLS

campos_buenos = [c for c in df_fe.columns if c not in EXCLUDE_COLS]

# ── Encode categoricals as int codes for LightGBM ────────────────────────────
CAT_COLS = ['country_short', 'pronoun_short', 'genre_short']

for col in CAT_COLS:
    df_fe[col] = df_fe[col].astype('category').cat.codes.astype(float)
    df_fe.loc[df_fe[col] == -1, col] = np.nan   # -1 = missing → NaN

# ── Convert nullable booleans to float (NaN-safe) ────────────────────────────
BOOL_COLS = ['band', 'muerto_disuelto', 'major_record_label']
for col in BOOL_COLS:
    df_fe[col] = df_fe[col].astype(float)

# ── Final feature matrix ──────────────────────────────────────────────────────
X_all = df_fe[campos_buenos].astype(float).values

print(f"campos_buenos : {len(campos_buenos)} features")
print(f"X_all shape   : {X_all.shape}")
print(f"y_all shape   : {y_all.shape}")
print(f"\nCategorical features passed to LightGBM: {CAT_COLS}")

# Quick null audit
null_pct = pd.DataFrame(X_all, columns=campos_buenos).isna().mean().sort_values(ascending=False)
print("\nTop-10 null rates:")
print(null_pct.head(10).round(3).to_string())

## 3. PARAM — semillas y configuración del experimento

Traducción fiel del workflow APO del profesor (R → Python).  
Todas las semillas se extraen de números primos para maximizar independencia estadística.

In [ ]:
# ── 3a. PARAM global ─────────────────────────────────────────────────────────
PARAM = {}
PARAM['experimento']       = 'lgbm-001'
PARAM['semilla_primigenia'] = 102191       # mismo que el profesor

# Bayesian Optimisation
PARAM['BO'] = {
    'iteraciones'  : 100,
    'ksemillerio'  : 5,    # seeds promediados por repetición (ensemble interno)
    'repe'         : 5,    # repeticiones independientes por trial
}

# Producción final (APO)
PARAM['train_final'] = {
    'APO'        : 10,   # repeticiones APO (anti-overfit)
    'ksemillerio': 10,   # seeds por repetición APO
}

# Válvula anti-overfitting (equivalente exacto del profesor)
assert (PARAM['train_final']['APO'] * PARAM['train_final']['ksemillerio'] >= 100
        and PARAM['train_final']['APO'] >= 5), "No quiera overfitear"

print("PARAM OK")
print(f"  BO        : {PARAM['BO']['iteraciones']} trials × "
      f"{PARAM['BO']['repe']} reps × {PARAM['BO']['ksemillerio']} seeds = "
      f"{PARAM['BO']['iteraciones'] * PARAM['BO']['repe'] * PARAM['BO']['ksemillerio']} fits")
print(f"  Producción: {PARAM['train_final']['APO']} APO × "
      f"{PARAM['train_final']['ksemillerio']} seeds = "
      f"{PARAM['train_final']['APO'] * PARAM['train_final']['ksemillerio']} fits")

In [ ]:
# ── 3b. Pool de primos + distribución de semillas ────────────────────────────
# Equivalente exacto de generate_primes(100000, 1000000) + sample() del profesor.
# Se usa criba de Eratóstenes (sin dependencias externas).

def sieve_primes(min_val, max_val):
    sieve = bytearray([1]) * (max_val + 1)
    sieve[0] = sieve[1] = 0
    for i in range(2, int(max_val**0.5) + 1):
        if sieve[i]:
            sieve[i*i::i] = bytearray(len(range(i*i, max_val + 1, i)))
    return [i for i in range(max(2, min_val), max_val + 1) if sieve[i]]

primos = sieve_primes(100_000, 1_000_000)
print(f"Primos disponibles [100K, 1M]: {len(primos):,}")

# Todas las semillas se sortean de una vez con la semilla primigenia
rng = np.random.default_rng(PARAM['semilla_primigenia'])

n_bo_seeds    = PARAM['BO']['ksemillerio'] * PARAM['BO']['repe']
n_final_seeds = PARAM['train_final']['APO'] * PARAM['train_final']['ksemillerio']

all_seeds = rng.choice(primos, size=n_bo_seeds + n_final_seeds, replace=False).tolist()

PARAM['BO']['semillas']           = all_seeds[:n_bo_seeds]
PARAM['train_final']['semillas']  = all_seeds[n_bo_seeds:]

print(f"\nSemillas BO ({n_bo_seeds}):      {PARAM['BO']['semillas']}")
print(f"Semillas finales ({n_final_seeds}) — primeras 5: {PARAM['train_final']['semillas'][:5]}")

In [ ]:
# ── 3c. Parámetros fijos de LightGBM ─────────────────────────────────────────
# Traducción directa de PARAM$lgbm$param_fijos del profesor.
# objective y metric cambian a regresión (el profesor usaba binary/custom).

PARAM['lgbm'] = {
    'param_fijos': {
        'objective'          : 'regression',     # regression_l2 = MSE loss
        'metric'             : 'rmse',
        'first_metric_only'  : True,
        'boost_from_average' : True,
        'feature_pre_filter' : False,
        'verbosity'          : -1,
        'force_row_wise'     : True,
        'extra_trees'        : False,
        'max_depth'          : -1,
        'min_gain_to_split'  : 0.0,
        'min_sum_hessian_in_leaf': 0.001,
        'lambda_l1'          : 0.0,
        'lambda_l2'          : 0.0,
        'bagging_fraction'   : 1.0,
        'is_unbalance'       : False,
        'max_bin'            : 31,
    }
}

print("Parámetros fijos LightGBM:")
for k, v in PARAM['lgbm']['param_fijos'].items():
    print(f"  {k:<28} = {v}")

## 4. Splits

**Outer split** — df_dev (70 %) / df_holdout (30 %):  
El holdout nunca se toca durante Optuna. Rol: *sanity check* de que el modelo aprendió algo real antes de interpretar los importances. Equivalente al `future = 202106` del profesor.

**Inner split** — df_bo_train / df_bo_val (70/30 dentro del dev):  
Todos los 100 trials de Optuna evalúan contra el mismo df_bo_val fijo.  
Equivalente al `testing = 202104` del profesor.

In [ ]:
# ── 4a. Outer split: dev (70%) / holdout (30%) ───────────────────────────────
# Estratificado por quintiles del target para preservar distribución.

y_bins = pd.qcut(pd.Series(y_all), q=5, labels=False, duplicates='drop')

idx = np.arange(len(X_all))
idx_dev, idx_holdout = train_test_split(
    idx,
    test_size=0.30,
    random_state=PARAM['semilla_primigenia'],
    stratify=y_bins,
)

X_dev,     X_holdout  = X_all[idx_dev],     X_all[idx_holdout]
y_dev,     y_holdout  = y_all[idx_dev],     y_all[idx_holdout]

print(f"dev      : {len(idx_dev):>5} filas ({len(idx_dev)/len(X_all)*100:.1f}%)")
print(f"holdout  : {len(idx_holdout):>5} filas ({len(idx_holdout)/len(X_all)*100:.1f}%)")

# ── 4b. Inner split (dentro de dev): bo_train / bo_val ───────────────────────
y_dev_bins = pd.qcut(pd.Series(y_dev), q=5, labels=False, duplicates='drop')

idx_dev_local = np.arange(len(X_dev))
idx_bo_train, idx_bo_val = train_test_split(
    idx_dev_local,
    test_size=0.30,
    random_state=PARAM['semilla_primigenia'],
    stratify=y_dev_bins,
)

X_bo_train, X_bo_val = X_dev[idx_bo_train], X_dev[idx_bo_val]
y_bo_train, y_bo_val = y_dev[idx_bo_train], y_dev[idx_bo_val]

print(f"\nbo_train : {len(idx_bo_train):>5} filas")
print(f"bo_val   : {len(idx_bo_val):>5} filas")

In [ ]:
# ── 4c. LightGBM Datasets ────────────────────────────────────────────────────
# Se construye dtrain_bo una sola vez. Todos los trials de Optuna lo reusan
# (equivalente al dtrain del profesor, precalculado antes del bucle BO).

cat_col_indices = [campos_buenos.index(c) for c in CAT_COLS]

dtrain_bo = lgb.Dataset(
    X_bo_train,
    label=y_bo_train,
    feature_name=campos_buenos,
    categorical_feature=cat_col_indices,
    free_raw_data=False,
)

print(f"dtrain_bo : {X_bo_train.shape[0]} filas × {X_bo_train.shape[1]} features")
print(f"categorical indices : {cat_col_indices} → {CAT_COLS}")

## 5. Optimización Bayesiana de Hiperparámetros (Optuna)

Traducción de `EstimarGanancia_lightgbm` del profesor al contexto de regresión.  
Métrica a minimizar: RMSE sobre `log1p(n_shows_24_25)` en `bo_val`.  
Espacio de búsqueda: escala logarítmica para `num_iterations`, `learning_rate`, `num_leaves`, `min_data_in_leaf` — igual al profesor.  
Restricción *forbidden*: `min_data_in_leaf × num_leaves ≤ n_bo_train`.

In [ ]:
# ── 5a. Función objetivo para Optuna ─────────────────────────────────────────

n_bo_train = X_bo_train.shape[0]

def bo_objective(trial):

    # ── Espacio de búsqueda (escala log igual al profesor) ────────────────
    num_iterations   = trial.suggest_int  ('num_iterations',   50,   2000, log=True)
    learning_rate    = trial.suggest_float('learning_rate',    1e-3, 0.3,  log=True)
    feature_fraction = trial.suggest_float('feature_fraction', 0.05, 1.0)
    num_leaves       = trial.suggest_int  ('num_leaves',       2,    512,  log=True)
    min_data_in_leaf = trial.suggest_int  ('min_data_in_leaf', 5,    n_bo_train // 2, log=True)

    # ── Restricción forbidden del profesor ────────────────────────────────
    if min_data_in_leaf * num_leaves > n_bo_train:
        return float('inf')

    params_trial = {
        **PARAM['lgbm']['param_fijos'],
        'num_iterations'   : num_iterations,
        'learning_rate'    : learning_rate,
        'feature_fraction' : feature_fraction,
        'num_leaves'       : num_leaves,
        'min_data_in_leaf' : min_data_in_leaf,
    }

    vrmse_reps = []

    # ── Loop de repe (medidas independientes, equivalente al profesor) ────
    for rep in range(PARAM['BO']['repe']):
        desde = rep * PARAM['BO']['ksemillerio']
        hasta = desde + PARAM['BO']['ksemillerio']
        rep_seeds = PARAM['BO']['semillas'][desde:hasta]

        vpred_acum = np.zeros(len(y_bo_val))

        # ── Loop semillerio (ensemble interno) ───────────────────────────
        for sem in rep_seeds:
            p = {**params_trial, 'seed': int(sem), 'verbosity': -1}
            model = lgb.train(p, dtrain_bo, valid_sets=None)
            vpred_acum += model.predict(X_bo_val)

        vpred_acum /= PARAM['BO']['ksemillerio']
        rmse_rep = np.sqrt(mean_squared_error(y_bo_val, vpred_acum))
        vrmse_reps.append(rmse_rep)

    mean_rmse = float(np.mean(vrmse_reps))
    trial.set_user_attr('rmse_std', float(np.std(vrmse_reps)))

    return mean_rmse

print("Función bo_objective definida.")

In [ ]:
# ── 5b. Crear y correr el estudio Optuna ─────────────────────────────────────
# TPESampler ≈ Kriging/EI de mlrMBO del profesor.
# El estudio es persistente en SQLite: si se interrumpe, se puede retomar
# (equivalente al mboContinue del profesor).

import os

STUDY_DB   = f"sqlite:///{PARAM['experimento']}_optuna.db"
STUDY_NAME = PARAM['experimento']

study = optuna.create_study(
    direction   = 'minimize',
    sampler     = optuna.samplers.TPESampler(seed=PARAM['semilla_primigenia']),
    study_name  = STUDY_NAME,
    storage     = STUDY_DB,
    load_if_exists = True,        # retoma si ya existe (= mboContinue)
)

n_done = len(study.trials)
n_remaining = max(0, PARAM['BO']['iteraciones'] - n_done)
print(f"Trials completados: {n_done}  |  pendientes: {n_remaining}")

if n_remaining > 0:
    study.optimize(
        bo_objective,
        n_trials  = n_remaining,
        show_progress_bar = True,
    )

print(f"\nMejor trial  : #{study.best_trial.number}")
print(f"Mejor RMSE   : {study.best_value:.6f}")
print(f"RMSE std     : {study.best_trial.user_attrs.get('rmse_std', 'n/a')}")
print(f"\nMejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k:<22} = {v}")

In [ ]:
# ── 5c. Log del estudio (equivalente al BO_log.txt del profesor) ─────────────
import csv, pathlib

log_path = pathlib.Path(f"{PARAM['experimento']}_bo_log.csv")

rows = []
for t in study.trials:
    if t.value is None:
        continue
    row = {'trial': t.number, 'rmse': t.value,
           'rmse_std': t.user_attrs.get('rmse_std', np.nan)}
    row.update(t.params)
    rows.append(row)

tb_bo = pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True)
tb_bo.to_csv(log_path, index=False)

print(f"Log guardado → {log_path}  ({len(tb_bo)} trials válidos)")
print("\nTop-5 trials:")
display(tb_bo.head())

## 6. Hiperparámetros óptimos — adaptación para producción

`min_data_in_leaf` se escala proporcionalmente al tamaño del dataset completo (4 913 filas), exactamente como hace el profesor al pasar de `dtrain` (BO) a `dtrain_final` (producción).

In [ ]:
# ── 6. Mejores HPs + escala min_data_in_leaf ─────────────────────────────────

best_params_bo = study.best_params.copy()

# El profesor escala min_data_in_leaf al pasar de bo_train a all_data.
# Aquí: all_data = 4913 filas  |  bo_train = n_bo_train filas.
scale_factor = len(X_all) / n_bo_train
mdil_original = best_params_bo['min_data_in_leaf']
mdil_scaled   = max(1, int(round(mdil_original * scale_factor)))

best_params_final = {
    **PARAM['lgbm']['param_fijos'],
    **best_params_bo,
    'min_data_in_leaf': mdil_scaled,
}

print(f"min_data_in_leaf  BO → final : {mdil_original} → {mdil_scaled}  "
      f"(escala ×{scale_factor:.2f})")
print(f"\nParámetros finales completos:")
for k, v in best_params_final.items():
    print(f"  {k:<28} = {v}")

PARAM['train_final']['param_mejores'] = best_params_final

## 7. Producción APO — modelos finales (todos los 4 913 artistas)

Para **explicabilidad**, los modelos de producción se entrenan sobre el dataset completo (no solo el 70% dev).  
Estructura APO = 10 repeticiones × 10 seeds = 100 modelos.  
Cada repetición promedía las predicciones de sus 10 seeds → 10 vectores de predicción independientes.  
La variabilidad entre las 10 repeticiones mide la estabilidad del resultado.

In [ ]:
# ── 7a. Dataset completo para producción ─────────────────────────────────────

dtrain_full = lgb.Dataset(
    X_all,
    label=y_all,
    feature_name=campos_buenos,
    categorical_feature=cat_col_indices,
    free_raw_data=False,
)
print(f"dtrain_full : {X_all.shape[0]} filas × {X_all.shape[1]} features")

In [ ]:
# ── 7b. Loop APO: 10 repeticiones × 10 seeds ─────────────────────────────────
# Estructura idéntica al profesor (cells 88-93 del notebook R).
# Se graba cada modelito en disco para poder recargar sin re-entrenar.

import os
modelitos_dir = pathlib.Path(f"modelitos_{PARAM['experimento']}")
modelitos_dir.mkdir(exist_ok=True)

APO          = PARAM['train_final']['APO']
K_SEM        = PARAM['train_final']['ksemillerio']
final_seeds  = PARAM['train_final']['semillas']
param_prod   = PARAM['train_final']['param_mejores'].copy()

# Almacenamos: para cada APO run, predicciones sobre holdout + importancias
apo_preds_holdout = np.zeros((APO, len(X_holdout)))
apo_importances   = []                       # lista de dicts gain por run

for vapo in range(APO):
    desde   = vapo * K_SEM
    hasta   = desde + K_SEM
    semillas_run = final_seeds[desde:hasta]

    vpred_full_acum    = np.zeros(X_all.shape[0])
    vpred_holdout_acum = np.zeros(len(X_holdout))
    impo_run           = np.zeros(len(campos_buenos))

    for sem in semillas_run:
        arch = modelitos_dir / f"mod_{sem}.txt"

        if arch.exists():
            m = lgb.Booster(model_file=str(arch))
        else:
            p = {**param_prod, 'seed': int(sem), 'verbosity': -1}
            m = lgb.train(p, dtrain_full)
            m.save_model(str(arch))

        vpred_holdout_acum += m.predict(X_holdout)
        impo_run           += m.feature_importance(importance_type='gain')

    apo_preds_holdout[vapo] = vpred_holdout_acum / K_SEM
    apo_importances.append(impo_run / K_SEM)

    rmse_run = np.sqrt(mean_squared_error(y_holdout, apo_preds_holdout[vapo]))
    print(f"APO run {vapo+1:2d}/{APO}  RMSE holdout = {rmse_run:.6f}")

print(f"\nModelos guardados en: {modelitos_dir}/")

## 8. Sanity check en holdout

RMSE ± SD sobre las 10 repeticiones APO.  
**Interpretación**: el holdout no mide generalización operativa sino que confirma que el modelo aprendió señal real antes de interpretar los importances.  
Si el RMSE es razonable (y la SD entre repeticiones es baja), los importances y SHAP values son confiables.

In [ ]:
# ── 8a. RMSE por APO run + resumen (equivalente a mganancias del profesor) ────

rmse_apo = np.array([
    np.sqrt(mean_squared_error(y_holdout, apo_preds_holdout[i]))
    for i in range(APO)
])

print("RMSE por APO run (escala log1p):")
for i, r in enumerate(rmse_apo):
    print(f"  run {i+1:2d}: {r:.6f}")

print(f"\nRMSE medio : {rmse_apo.mean():.6f}")
print(f"RMSE SD    : {rmse_apo.std():.6f}")

# Guardar tabla APO (= tb_apo.txt del profesor)
tb_apo = pd.DataFrame({'rmse_holdout': rmse_apo})
tb_apo.to_csv(f"{PARAM['experimento']}_tb_apo.csv", index=False)

# ── Predicciones promediadas (media APO) ─────────────────────────────────────
y_pred_mean = apo_preds_holdout.mean(axis=0)   # promedio de los 10 runs

# Comparación en escala original (expm1)
y_holdout_orig = np.expm1(y_holdout)
y_pred_orig    = np.expm1(y_pred_mean)

mae_orig  = np.mean(np.abs(y_holdout_orig - y_pred_orig))
rmse_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_pred_orig))
print(f"\nEscala original (shows reales):")
print(f"  MAE  = {mae_orig:.2f} shows")
print(f"  RMSE = {rmse_orig:.2f} shows")

## 9. Importancia de variables

Importancia tipo `gain` promediada sobre los 100 modelos (10 APO × 10 seeds), agrupada por dimensión.  
`gain` mide la reducción total de MSE que aporta cada feature — más estable que `split` para comparar dimensiones.

In [ ]:
# ── 9a. Importancia promedio + SD entre runs ──────────────────────────────────

impo_matrix = np.vstack(apo_importances)           # shape (APO, n_features)
impo_mean   = impo_matrix.mean(axis=0)
impo_std    = impo_matrix.std(axis=0)

tb_impo = pd.DataFrame({
    'feature'  : campos_buenos,
    'gain_mean': impo_mean,
    'gain_std' : impo_std,
}).sort_values('gain_mean', ascending=False).reset_index(drop=True)

# Asignar dimensión a cada feature
def asignar_dim(f):
    if f in streaming_cols:         return 'streaming'
    if f in playlist_cols:          return 'playlists'
    if f in social_cols:            return 'redes_sociales'
    if f in youtube_cols:           return 'youtube'
    if f in ratio_cols:             return 'ratios'
    return 'metadata'

tb_impo['dimension'] = tb_impo['feature'].apply(asignar_dim)

# Normalizar a porcentaje
tb_impo['gain_pct'] = tb_impo['gain_mean'] / tb_impo['gain_mean'].sum() * 100

tb_impo.to_csv(f"{PARAM['experimento']}_importances.csv", index=False)

print("Top-20 features por gain:")
display(tb_impo[['feature', 'dimension', 'gain_pct', 'gain_std']].head(20).round(3))

In [ ]:
# ── 9b. Importancia por dimensión ────────────────────────────────────────────

impo_dim = (
    tb_impo.groupby('dimension')
    .agg(gain_total=('gain_mean', 'sum'), n_vars=('feature', 'count'))
    .assign(gain_pct=lambda d: d['gain_total'] / d['gain_total'].sum() * 100)
    .sort_values('gain_pct', ascending=False)
)
print("Importancia por dimensión:")
display(impo_dim.round(2))

# ── Gráfico horizontal ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: top-20 features
top20 = tb_impo.head(20).iloc[::-1]
axes[0].barh(top20['feature'], top20['gain_pct'],
             xerr=top20['gain_std'] / tb_impo['gain_mean'].sum() * 100,
             color='#1F4E79', ecolor='gray', capsize=3)
axes[0].set_xlabel("Gain (%)")
axes[0].set_title("Top-20 features por importancia (gain)")

# Panel derecho: por dimensión
colors = {'streaming':'#1F4E79','playlists':'#2E86AB','redes_sociales':'#A23B72',
          'youtube':'#F18F01','ratios':'#C73E1D','metadata':'#6c757d'}
dim_plot = impo_dim.reset_index()
axes[1].barh(dim_plot['dimension'], dim_plot['gain_pct'],
             color=[colors.get(d, '#999') for d in dim_plot['dimension']])
axes[1].set_xlabel("Gain (%)")
axes[1].set_title("Importancia por dimensión")

plt.tight_layout()
plt.savefig(f"{PARAM['experimento']}_importances.png", dpi=150, bbox_inches='tight')
plt.show()